## This notebook works best with GPU

GPU accelerates inference timing comparisons. CPU works but results will be slower.

**Runtime > Change Runtime Type > T4 GPU**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/msds-marketing-analytics/colab-notebooks/blob/main/LLMs/MSDSTextClassification_ProductionDeploymentScaling.ipynb)

# Serving LLM Classifiers in Production

Training a model is only half the work. To be useful, it needs to serve
predictions reliably, quickly, and at scale. This notebook covers the
practical skills for wrapping a classifier in an API, measuring its
performance, monitoring it, and preparing it for deployment.

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Build a FastAPI service** around an LLM classifier and test it with real requests
2. **Measure inference performance** — latency per request and throughput with batching
3. **Implement request monitoring** connected to actual inference, not hardcoded values
4. **Optimize serving with FP16** and measure the speed/accuracy tradeoff
5. **Write deployment configurations** (Dockerfile, docker-compose) for containerized serving
6. **Understand deployment considerations** — scaling, cloud options, and operational concerns

## What We Can and Cannot Do in Colab

We cannot start a persistent server, build Docker images, or deploy to the
cloud from a notebook. What we **can** do is build the API, test it with
FastAPI's `TestClient` (which simulates HTTP requests without a running
server), and measure real performance. The containerization and deployment
sections are reference material — correctly structured configs you would
use outside Colab.

In [ ]:
!pip install -q fastapi httpx transformers torch matplotlib

In [ ]:
import torch
import numpy as np
import time
from datetime import datetime
from typing import List, Optional, Dict, Any
from transformers import pipeline, AutoModelForSequenceClassification, AutoTokenizer
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

device_id = 0 if torch.cuda.is_available() else -1
device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print(f"Device: {device_name}")

---
## Part 1: Building and Testing a Classification API

We wrap `distilbert-base-uncased-finetuned-sst-2-english` in a FastAPI
application with endpoints for classification, health checks, and metrics.
Then we test it using `TestClient`, which sends real HTTP requests to the
app without needing a running server.

In [ ]:
# Load the sentiment classifier
model_name = "distilbert-base-uncased-finetuned-sst-2-english"
classifier = pipeline(
    "sentiment-analysis",
    model=model_name,
    device=device_id
)

# Quick sanity check
test = classifier("This is a great product!")[0]
print(f"Model loaded: {model_name}")
print(f"Test: '{test['label']}' (score: {test['score']:.3f})")

In [ ]:
# --- Metrics collector (used by the API) ---
class RequestMetrics:
    """Collects latency and error metrics from real requests."""
    def __init__(self):
        self.latencies = []
        self.errors = 0
        self.start_time = time.time()

    def record(self, latency: float):
        self.latencies.append(latency)

    def record_error(self):
        self.errors += 1

    def summary(self) -> dict:
        n = len(self.latencies)
        if n == 0:
            return {"requests": 0}
        sorted_lat = sorted(self.latencies)
        return {
            "requests": n,
            "errors": self.errors,
            "uptime_seconds": round(time.time() - self.start_time, 1),
            "mean_latency": round(np.mean(sorted_lat), 4),
            "p50_latency": round(sorted_lat[int(n * 0.50)], 4),
            "p95_latency": round(sorted_lat[min(int(n * 0.95), n - 1)], 4),
            "p99_latency": round(sorted_lat[min(int(n * 0.99), n - 1)], 4),
            "error_rate": round(self.errors / n, 4) if n > 0 else 0,
        }

metrics = RequestMetrics()

# --- FastAPI application ---
app = FastAPI(title="Sentiment Classification API", version="1.0.0")

class ClassifyRequest(BaseModel):
    text: str

class ClassifyResponse(BaseModel):
    prediction: str
    confidence: float
    latency_ms: float

@app.post("/classify", response_model=ClassifyResponse)
def classify_endpoint(request: ClassifyRequest):
    start = time.time()
    result = classifier(request.text, truncation=True, max_length=512)[0]
    latency = time.time() - start
    metrics.record(latency)
    return ClassifyResponse(
        prediction=result["label"],
        confidence=round(result["score"], 4),
        latency_ms=round(latency * 1000, 1)
    )

@app.get("/health")
def health():
    return {"status": "healthy", "model": model_name,
            "timestamp": datetime.now().isoformat()}

@app.get("/metrics")
def get_metrics():
    return metrics.summary()

print("FastAPI app defined with endpoints: /classify, /health, /metrics")

In [ ]:
# Test the API using FastAPI's TestClient (no server needed)
client = TestClient(app)

# Health check
response = client.get("/health")
print(f"GET /health -> {response.status_code}")
print(f"  {response.json()}")

# Classification
test_texts = [
    "This movie was absolutely fantastic! Best I've seen all year.",
    "Terrible waste of time. The plot made no sense.",
    "It was okay, nothing special but not terrible either.",
]

print(f"\nPOST /classify")
for text in test_texts:
    response = client.post("/classify", json={"text": text})
    r = response.json()
    print(f"  '{text[:50]}...'")
    print(f"    -> {r['prediction']} ({r['confidence']:.3f}) in {r['latency_ms']:.0f}ms")

# Metrics after those requests
print(f"\nGET /metrics")
print(f"  {client.get('/metrics').json()}")

---
## Part 2: Measuring Inference Performance

In production, you need to know how fast your model serves predictions and
how throughput scales with load. We measure single-request latency, then
compare throughput when calling the model directly with different batch sizes
versus calling through the API one request at a time.

In [ ]:
from datasets import load_dataset

# Load 200 test texts for benchmarking
bench_data = load_dataset("imdb", split="test").shuffle(seed=42).select(range(200))
bench_texts = list(bench_data["text"])

# --- API throughput (one request at a time via TestClient) ---
n_api = 50  # fewer for API since it's slower per-request
start = time.time()
for text in bench_texts[:n_api]:
    client.post("/classify", json={"text": text})
api_time = time.time() - start
api_throughput = n_api / api_time

print(f"API throughput ({n_api} requests):")
print(f"  Total time: {api_time:.1f}s")
print(f"  Throughput: {api_throughput:.1f} req/s")
print(f"  Avg latency: {api_time/n_api*1000:.0f}ms per request")

# --- Direct pipeline throughput with batching ---
batch_sizes = [1, 8, 16, 32]
batch_results = []

for bs in batch_sizes:
    start = time.time()
    _ = classifier(bench_texts[:200], batch_size=bs, truncation=True, max_length=512)
    elapsed = time.time() - start
    throughput = 200 / elapsed
    batch_results.append({"batch_size": bs, "time": elapsed, "throughput": throughput})
    print(f"\nDirect pipeline (batch_size={bs}, 200 texts):")
    print(f"  Total time: {elapsed:.1f}s")
    print(f"  Throughput: {throughput:.1f} texts/s")

In [ ]:
# Visualize throughput scaling
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bs_labels = [str(r["batch_size"]) for r in batch_results]
throughputs = [r["throughput"] for r in batch_results]
times = [r["time"] for r in batch_results]

axes[0].bar(bs_labels, throughputs, color="steelblue", alpha=0.7)
axes[0].set_xlabel("Batch Size")
axes[0].set_ylabel("Throughput (texts/sec)")
axes[0].set_title("Throughput vs Batch Size (200 texts)")
for i, v in enumerate(throughputs):
    axes[0].text(i, v + 0.5, f"{v:.1f}", ha="center", fontsize=9)

axes[1].bar(bs_labels, times, color="salmon", alpha=0.7)
axes[1].set_xlabel("Batch Size")
axes[1].set_ylabel("Total Time (seconds)")
axes[1].set_title("Processing Time vs Batch Size (200 texts)")
for i, v in enumerate(times):
    axes[1].text(i, v + 0.2, f"{v:.1f}s", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

best = max(batch_results, key=lambda r: r["throughput"])
worst = min(batch_results, key=lambda r: r["throughput"])
print(f"Best: batch_size={best['batch_size']} ({best['throughput']:.1f} texts/s)")
print(f"Worst: batch_size={worst['batch_size']} ({worst['throughput']:.1f} texts/s)")
print(f"\nFor this small model ({model_name}), batching may show modest or no")
print(f"improvement — the model is fast enough that batch coordination overhead")
print(f"can dominate. Batching yields much larger gains with bigger models (7B+)")
print(f"where GPU utilization per request is low and there is more idle compute")
print(f"to fill.")

---
## Part 3: Monitoring Real Requests

The API has been collecting latency metrics from every request we've made.
Let's send a larger batch of requests through the API and examine what the
monitoring data tells us.

In [ ]:
# Reset metrics and send 100 requests through the API
metrics = RequestMetrics()
# Rebind so the app uses the new metrics object
app.dependency_overrides = {}

print("Sending 100 requests through the API...")
for text in bench_texts[:100]:
    response = client.post("/classify", json={"text": text})

summary = metrics.summary()
print(f"\nMonitoring Summary ({summary['requests']} requests):")
print(f"  Mean latency:  {summary['mean_latency']*1000:.0f}ms")
print(f"  P50 latency:   {summary['p50_latency']*1000:.0f}ms")
print(f"  P95 latency:   {summary['p95_latency']*1000:.0f}ms")
print(f"  P99 latency:   {summary['p99_latency']*1000:.0f}ms")
print(f"  Error rate:    {summary['error_rate']}")
print(f"  Uptime:        {summary['uptime_seconds']}s")

In [ ]:
# Visualize latency distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

latencies_ms = [l * 1000 for l in metrics.latencies]

axes[0].hist(latencies_ms, bins=20, color="steelblue", alpha=0.7, edgecolor="white")
axes[0].axvline(summary["p50_latency"] * 1000, color="green", linestyle="--",
                label=f"P50: {summary['p50_latency']*1000:.0f}ms")
axes[0].axvline(summary["p95_latency"] * 1000, color="orange", linestyle="--",
                label=f"P95: {summary['p95_latency']*1000:.0f}ms")
axes[0].axvline(summary["p99_latency"] * 1000, color="red", linestyle="--",
                label=f"P99: {summary['p99_latency']*1000:.0f}ms")
axes[0].set_xlabel("Latency (ms)")
axes[0].set_ylabel("Count")
axes[0].set_title("Latency Distribution (100 API requests)")
axes[0].legend()

# Latency over time (request order)
axes[1].plot(latencies_ms, color="steelblue", alpha=0.7, linewidth=0.8)
axes[1].axhline(summary["mean_latency"] * 1000, color="red", linestyle="--",
                label=f"Mean: {summary['mean_latency']*1000:.0f}ms")
axes[1].set_xlabel("Request Number")
axes[1].set_ylabel("Latency (ms)")
axes[1].set_title("Latency Over Time")
axes[1].legend()

plt.tight_layout()
plt.show()

print("In production, these metrics would feed into Prometheus/Grafana")
print("or a cloud monitoring service (CloudWatch, Cloud Monitoring, etc.).")

---
## Part 4: Model Optimization for Serving

One of the simplest optimizations for production serving is running inference
in **FP16** (half precision) instead of FP32. This halves memory usage and
often speeds up inference on GPUs with Tensor Cores, with negligible accuracy
loss for most classification tasks.

Let's measure the actual impact.

In [ ]:
# Load the same model in FP32 and FP16 and compare
print("Loading model in FP32...")
model_fp32 = AutoModelForSequenceClassification.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
if torch.cuda.is_available():
    model_fp32 = model_fp32.cuda()
model_fp32.eval()

print("Loading model in FP16...")
model_fp16 = AutoModelForSequenceClassification.from_pretrained(
    model_name, torch_dtype=torch.float16
)
if torch.cuda.is_available():
    model_fp16 = model_fp16.cuda()
model_fp16.eval()

# Compare model sizes
fp32_size = sum(p.numel() * p.element_size() for p in model_fp32.parameters())
fp16_size = sum(p.numel() * p.element_size() for p in model_fp16.parameters())

print(f"\nModel memory:")
print(f"  FP32: {fp32_size / 1024**2:.1f} MB")
print(f"  FP16: {fp16_size / 1024**2:.1f} MB")
print(f"  Reduction: {(1 - fp16_size/fp32_size)*100:.0f}%")

In [ ]:
# Benchmark inference speed: FP32 vs FP16
n_bench = 100
bench_subset = bench_texts[:n_bench]

def time_inference(model, texts, label):
    """Time inference on a list of texts."""
    device = next(model.parameters()).device
    start = time.time()
    for text in texts:
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
    elapsed = time.time() - start
    print(f"  {label}: {elapsed:.1f}s ({n_bench/elapsed:.1f} texts/s, {elapsed/n_bench*1000:.0f}ms avg)")
    return elapsed

print(f"Inference speed ({n_bench} texts):")
t_fp32 = time_inference(model_fp32, bench_subset, "FP32")
t_fp16 = time_inference(model_fp16, bench_subset, "FP16")

print(f"\nFP16 speedup: {t_fp32/t_fp16:.2f}x")

# Verify accuracy is preserved
sample_texts = bench_texts[:20]
pipe_fp32 = pipeline("sentiment-analysis", model=model_fp32, tokenizer=tokenizer, device=device_id)
pipe_fp16 = pipeline("sentiment-analysis", model=model_fp16, tokenizer=tokenizer, device=device_id)

preds_fp32 = [r["label"] for r in pipe_fp32(sample_texts, truncation=True, max_length=512)]
preds_fp16 = [r["label"] for r in pipe_fp16(sample_texts, truncation=True, max_length=512)]
agreement = sum(a == b for a, b in zip(preds_fp32, preds_fp16)) / len(preds_fp32)

print(f"Prediction agreement (FP32 vs FP16): {agreement*100:.0f}% on {len(sample_texts)} examples")

---
## Part 5: Containerization for Deployment

To deploy outside Colab, the service needs to be packaged in a Docker
container. The files below are reference configurations — they cannot be
built or run inside Colab, but they are what you would use to deploy the
API we built in Part 1 to any server or cloud platform.

### How it works

1. The **Dockerfile** defines the environment: Python, dependencies, model
   download, and the uvicorn command to start the API.
2. **docker-compose** adds a health check and GPU reservation.
3. You build once, then deploy the image anywhere — EC2, GKE, Azure Container
   Instances, or your own server.

In [ ]:
# Reference Dockerfile for the classification API
dockerfile = '''\
FROM python:3.10-slim

WORKDIR /app

# Install dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Download the model at build time (so it's baked into the image)
RUN python -c "from transformers import pipeline; \\
    pipeline('sentiment-analysis', model='distilbert-base-uncased-finetuned-sst-2-english')"

# Copy application code
COPY app.py .

# Non-root user for security
RUN useradd --create-home appuser && chown -R appuser:appuser /app
USER appuser

EXPOSE 8000

# Start the API
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
'''

docker_compose = '''\
services:
  classifier:
    build: .
    ports:
      - "8000:8000"
    deploy:
      resources:
        reservations:
          devices:
            - driver: nvidia
              count: 1
              capabilities: [gpu]
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8000/health"]
      interval: 30s
      timeout: 10s
      retries: 3
'''

print("=== Dockerfile ===")
print(dockerfile)
print("=== docker-compose.yml ===")
print(docker_compose)
print("To deploy: docker compose up --build")
print("The API would be available at http://localhost:8000/docs")

---
## Part 6: Deployment Considerations

### Scaling Strategies

| Strategy | How It Works | When to Use |
|----------|-------------|-------------|
| **Single instance** | One container, one GPU | Prototyping, low traffic (<10 req/s) |
| **Horizontal scaling** | Multiple identical containers behind a load balancer | Moderate traffic, predictable load |
| **Auto-scaling** | Cloud platform adds/removes instances based on CPU/GPU utilization | Variable traffic, cost optimization |
| **Batched inference** | Collect requests into batches before running inference | High throughput, latency-tolerant |

### Cloud Platform Options

All major platforms support GPU-accelerated container serving:

- **AWS**: SageMaker Endpoints (managed) or ECS/EKS with GPU instances
- **GCP**: Vertex AI (managed) or GKE with GPU node pools
- **Azure**: Azure ML Endpoints (managed) or AKS with GPU nodes

Managed services (SageMaker, Vertex AI) handle auto-scaling, health checks,
and model versioning for you. Self-managed container orchestration (EKS, GKE,
AKS) gives more control but requires more operational work.

### Operational Checklist

Before deploying to production:

1. **Health checks**: The `/health` endpoint lets the load balancer and
   orchestrator know the service is ready to accept traffic.
2. **Metrics export**: In production, the `/metrics` data would feed into
   Prometheus + Grafana or a cloud-native monitoring service.
3. **Request validation**: The Pydantic models in Part 1 reject malformed
   requests automatically. Add input length limits for safety.
4. **Rate limiting**: Protect the GPU from being overwhelmed. FastAPI
   middleware or an API gateway can enforce limits.
5. **Model versioning**: Tag Docker images with model versions. Roll back
   by redeploying the previous image.
6. **Logging**: Structured logging (JSON) makes it easy to search and
   alert on errors in production.

---
## Key Takeaways

1. **FastAPI + TestClient** lets you build and test a real API without
   deploying a server. The same app code runs in production with `uvicorn`.

2. **Batching helps — but not always.** For small, fast models, batching
   overhead can offset the gains. The benefit scales with model size: for
   7B+ parameter models where each request underutilizes the GPU, batching
   fills idle compute and dramatically improves throughput.

3. **Monitor real requests, not synthetic ones.** The metrics collector in
   this notebook tracked actual inference latencies. In production, export
   these to Prometheus/Grafana.

4. **FP16 is a free lunch for serving.** Half the memory, faster inference,
   near-identical predictions. Enable it unless you have a specific reason not to.

5. **Containerize for portability.** A Docker image packages the model,
   dependencies, and API together. Deploy anywhere with `docker run`.

## Exercises

1. **Add a batch endpoint**: Create a `POST /classify/batch` endpoint that
   accepts a list of texts and classifies them all at once using the pipeline's
   batch_size parameter. Compare its throughput with the single-text endpoint.

2. **Error handling**: Send malformed requests (empty text, very long text,
   non-string input) and see how the API handles them. Add explicit validation.

3. **Model swapping**: Add an endpoint that lets you switch between different
   models at runtime. How do you handle requests that arrive during the swap?

4. **Latency budgets**: If your SLA requires P95 latency under 100ms, what is
   the maximum input length you can accept? Measure latency vs. input length.

5. **Deploy locally**: Take the Dockerfile from Part 5, create the
   `requirements.txt` and `app.py`, and deploy the service on your own machine.
   Test it with `curl` or the Swagger UI at `/docs`.